In [1]:
import torch
from torchnmf.nmf import NMF
import numpy as np
import pandas as pd
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [2]:
chr = "chr20"

encoding_EPIC_V1 = pd.read_csv(
    "../res/EPIC_V1_V2_encoded/EPIC_V1_{}_4000.csv".format(chr), index_col=0)
encoding_EPIC_V2 = pd.read_csv(
    "../res/EPIC_V1_V2_encoded/EPIC_V2_{}_4000.csv".format(chr), index_col=0)

out_path = "../res/EPIC_V1_V2_NMF/"

In [3]:
assert encoding_EPIC_V1.shape[0] == encoding_EPIC_V2.shape[0]
assert encoding_EPIC_V1.shape[1] == encoding_EPIC_V2.shape[1]

print("Encoding size", encoding_EPIC_V1.shape[1])
rank = 500
rank = encoding_EPIC_V1.shape[1] * 4
print("Rank", rank)
alpha = 1
beta = 4
l1_ratio = 0

Encoding size 2560
Rank 10240


In [4]:
train_samples = pd.read_csv(
    "../tmp/EPIC_V1_V2/train_samples.txt", header=None)[0].values
val_samples = pd.read_csv(
    "../tmp/EPIC_V1_V2/val_samples.txt", header=None)[0].values
test_samples = pd.read_csv(
    "../tmp/EPIC_V1_V2/test_samples.txt", header=None)[0].values

encoding_EPIC_V1_test = encoding_EPIC_V1.loc[test_samples]
encoding_EPIC_V2_test = encoding_EPIC_V2.loc[test_samples]

In [5]:
# encoding_450k_train = encoding_450k.drop(test_samples)
# encoding_EPIC_train = encoding_EPIC.drop(test_samples)
encoding_EPIC_V1_train = encoding_EPIC_V1.loc[train_samples]
encoding_EPIC_V2_train = encoding_EPIC_V2.loc[train_samples]

In [6]:
# NMF
# concat encoding_450k_train and encoding_EPIC_train
encoding_train = pd.concat([encoding_EPIC_V1_train, encoding_EPIC_V2_train], axis=1).to_numpy()
encoding_train = torch.from_numpy(encoding_train).float()

In [7]:
# First round of NMF
model = NMF(encoding_train.shape, rank=rank).cuda()
model.fit(encoding_train.cuda(), tol=1e-10, max_iter=10000,
          verbose=True, alpha=alpha, beta=beta, l1_ratio=l1_ratio)

  0%|          | 20/10000 [00:00<00:50, 196.15it/s, loss=5.34]


20

In [8]:
W = model.W.cpu().detach().numpy()
H = model.H.cpu().detach().numpy()

In [9]:
# compute the reconstruction error
encoding_train_reconstruct = np.matmul(H, W.T)
encoding_train_reconstruct.shape

(35, 5120)

In [10]:
np.abs((encoding_train.numpy() - encoding_train_reconstruct)).mean()

0.02596443

In [11]:
W1 = W[0:W.shape[0]//2, :]
W2 = W[W.shape[0]//2:, :]

In [12]:
# encoding_450k_test.shape
encoding_EPIC_V1_test.shape

(5, 2560)

In [13]:
# Second round of NMF
# 450k -> EPIC
model_2 = NMF(encoding_EPIC_V1_test.shape, W=W1, trainable_W=False, rank=rank).cuda()

In [14]:
model_2.fit(torch.from_numpy(
    encoding_EPIC_V1_test.to_numpy()).float().cuda(), tol=1e-10, max_iter=10000, verbose=True, alpha=alpha, beta=beta, l1_ratio=l1_ratio)

  0%|          | 20/10000 [00:00<00:22, 451.01it/s, loss=3.28]


20

In [15]:
# reconstruct EPIC from 450k
encoding_EPIC_V2_reconstruct = np.matmul(model_2.H.cpu().detach().numpy(), W2.T)

In [16]:
# compute the reconstruction error
np.abs((encoding_EPIC_V2_test.to_numpy() - encoding_EPIC_V2_reconstruct)).mean()

0.08235790585756808

In [17]:
# save the results
df = pd.DataFrame(encoding_EPIC_V2_reconstruct, index=encoding_EPIC_V2_test.index, columns=encoding_EPIC_V2_test.columns)

In [18]:
df.to_csv(out_path + "/{}_reconstruct_new.csv".format(chr))

In [19]:
np.sqrt(((encoding_EPIC_V2_test.to_numpy() - encoding_EPIC_V2_reconstruct)**2).mean())

0.09834306800712288